In [1]:
from earthdaily.earthone.catalog import Product, Image, properties as p
import earthdaily.earthone as eo

In [25]:
import geopandas as gpd
from datetime import datetime
import numpy as np
import pandas as pd

In [3]:
auth = eo.auth.Auth.get_default_auth()
org = auth.payload["org"]
user_hash = auth.namespace

In [4]:
run_id =   user_hash + ":" + datetime.utcnow().strftime("%Y%m%d")
func_name = f"Water Quality Monitor {run_id}"
func_name

'Water Quality Monitor 3935a4c94291fae6f3fedfbaa29a292a8c5687b2:20260617'

In [5]:
auth = eo.auth.Auth.get_default_auth()
org = auth.payload["org"]
user_hash = auth.namespace

In [6]:
from utils import create_s2_surrogate

In [12]:
prod = Product.get(f"earthdaily:surrogate-s2:{run_id}")
images = prod.images().collect()
stack  = images.stack(["nir", "red", "green"], geocontext=images[0].geocontext)

In [26]:
def diff_stats(target_stack, baseline_ndti, dates=None, thresholds=(0.02, 0.05, 0.10, 0.15, 0.20, 0.30)):
    """
    target_stack:  shape (n_scenes, 3, y, x), band order [nir, red, green]
    baseline_ndti: shape (1, y, x) or (y, x)
    dates:         optional list of labels/dates, length n_scenes
    """
    baseline = baseline_ndti[0] if baseline_ndti.ndim == 3 else baseline_ndti

    nir   = target_stack[:, 0]
    red   = target_stack[:, 1]
    green = target_stack[:, 2]

    ndwi = (green - nir) / (green + nir)
    ndti = (red - green) / (red + green)
    ndti_water = np.where(ndwi > 0, ndti, np.nan)

    diff = ndti_water - baseline   # broadcasts (n_scenes, y, x) - (y, x) -> (n_scenes, y, x)

    if dates is None:
        dates = [f"scene_{i}" for i in range(target_stack.shape[0])]

    rows = []
    for i, label in enumerate(dates):
        d = diff[i]
        valid = np.sum(~np.isnan(d))
        row = {
            "label": label,
            "valid_pixels": valid,
            "mean": np.nanmean(d),
            "std": np.nanstd(d),
            "min": np.nanmin(d),
            "max": np.nanmax(d),
        }
        for t in thresholds:
            row[f"pct_above_{t}"] = 100 * np.nansum(d > t) / valid
        rows.append(row)

    return pd.DataFrame(rows), diff

In [27]:
baseline_prod = Product.get(f"earthdaily:turbidity-baseline:{run_id}")
baseline_ndti = baseline_prod.images().collect().mosaic(["ndti"], geocontext=images[0].geocontext)

In [29]:
df, diff  = diff_stats(stack, baseline_ndti, dates=list(images.each.acquired))
print(df.to_string(index=False))

                    label  valid_pixels     mean      std       min      max  pct_above_0.02  pct_above_0.05  pct_above_0.1  pct_above_0.15  pct_above_0.2  pct_above_0.3
2024-01-14 00:00:00+00:00        237921 0.167821 0.065980 -0.103489 0.547599       97.858113       95.628381      82.861118       61.699051      41.525548       0.560690
2024-03-14 00:00:00+00:00        238023 0.049909 0.069868 -0.352451 0.453212       62.180966       52.441151      33.714809        2.217853       0.445755       0.065960
2024-05-18 00:00:00+00:00        229131 0.232216 0.097788 -0.119379 0.611894       98.605165       95.537924      85.898896       77.192087      65.794240      36.086344
2024-06-07 00:00:00+00:00        238001 0.239828 0.079287 -0.121331 0.623062       99.779833       99.210928      96.366822       84.711409      64.551830      30.413738


In [ ]:
surrogate_s2_product = create_s2_surrogate(
    f"surrogate-s2:{run_id}",
    "Murray River Surrogate Sentinel-2 Product",
    "data/MurrayMouth.geojson"
)

In [ ]:
image = Image(
    name="murray-river-2022-12-07",
    product=surrogate_s2_product,
    acquired="2022-12-07"
)
upload = image.upload(
    ["data/murray_fcc_2022-12-07.tif"],
    overwrite=True,
)
upload.wait_for_completion()
upload.status

In [ ]:
os.remove("data/murray_fcc_2022-12-07")